# CS231 — So sánh các CNN trên Colab

Bật GPU. Upload `cs231_colab_bundle.zip` vào session storage `/content` qua Files → Upload. Gói chứa code, dataset và ResNet18 checkpoint; notebook không dùng Google Drive.

In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime.'
print(torch.cuda.get_device_name(0), torch.__version__)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=1.0.8', 'scikit-learn>=1.3', 'scikit-image>=0.22', 'pyyaml', 'pandas', 'joblib', 'opencv-python-headless'], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, json, hashlib
bundle = Path('/content/cs231_colab_bundle.zip')
if not bundle.exists():
    files.upload()
assert bundle.exists(), 'Upload cs231_colab_bundle.zip vào /content.'
work = Path('/content/cs231_comparison')
work.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(work)
os.chdir(work)
sys.path.insert(0, str(work))
manifest = json.loads((work/'bundle_manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((work/name).read_bytes()).hexdigest() == digest
print('Bundle verified:', manifest['source_commit'])


In [ ]:
import subprocess, sys
data = '/content/cs231_comparison/data/standard_split'
for model, freeze in [('light_cnn', '0'), ('resnet18', '3'), ('resnet50', '3'), ('mobilenetv4_conv_small', '3')]:
    subprocess.run([sys.executable, '-u', '-m', 'src.member2_deeplearning.train_cnn', '--model', model, '--epochs', '30', '--patience', '8', '--batch_size', '32', '--workers', '2', '--lr', '0.0002', '--label_smoothing', '0.05', '--data', data, '--output_root', '/content/cs231_comparison', '--freeze_epochs', freeze], check=True)


In [ ]:
# Benchmark all CNN checkpoints on the same held-out test split.
subprocess.run([sys.executable, 'benchmark_three_models.py'], check=True)
from pathlib import Path
print(Path('results/three_model_comparison/comparison.csv').read_text())


Kết quả lưu tại `/content/cs231_comparison/results/three_model_comparison/`. Tải thư mục này từ Files pane sau khi chạy xong. ResNet50 là backbone tối ưu bổ sung; checkpoint được chọn theo validation Macro F1.